# MetaXtractor - Complete Extraction Pipeline

This notebook provides a complete pipeline for extracting metadata from GitHub repositories using:
1. **GitHub API** - Repository information and README files
2. **Structured Files** - Citation.cff, setup.py, pyproject.toml, etc.
3. **NER Model** - Metadata extraction from preprocessed README text
4. **Priority-based Merging** - GitHub > Structured Files > Model extraction
5. **CodeMeta Output** - Final merged metadata in CodeMeta JSON format

## 1. Setup and Dependencies

In [45]:
# Install dependencies
!pip install torch scikit-learn transformers requests beautifulsoup4 pandas tqdm tomli pyyaml --quiet

import re
import json
import os
import base64
import logging
import requests
import yaml
import hashlib
import requests
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, Optional, List, Tuple

# ML/NER imports
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("MetaXtractor")

# Device setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {DEVICE}")

# Model directory
MODEL_DIR = "../../data/model"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


2025-11-18 13:44:12,310 - INFO - Using device: cpu


## 2. Configuration

In [46]:
# GitHub API configuration
GITHUB_TOKEN = input("Enter your GitHub token (or press Enter to skip): ").strip()
if not GITHUB_TOKEN:
    logger.warning("No GitHub token provided. API rate limits will apply.")
    HEADERS = {"Accept": "application/vnd.github.v3+json"}
else:
    HEADERS = {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept": "application/vnd.github.v3+json"
    }

# NER Model configuration
NER_THRESHOLD = 0.8
MAX_TOKENS = 480

LICENSE_URL_MAP = {
    'MIT': 'https://opensource.org/licenses/MIT',
    'MIT License': 'https://opensource.org/licenses/MIT',
    'Apache': 'https://opensource.org/licenses/Apache-2.0',
    'Apache 2.0': 'https://opensource.org/licenses/Apache-2.0',
    'Apache License 2.0': 'https://opensource.org/licenses/Apache-2.0',
    'Apache-2.0': 'https://opensource.org/licenses/Apache-2.0',
    'GPL': 'https://www.gnu.org/licenses/gpl-3.0.html',
    'GPL-3.0': 'https://www.gnu.org/licenses/gpl-3.0.html',
    'GNU General Public License': 'https://www.gnu.org/licenses/gpl-3.0.html',
    'BSD': 'https://opensource.org/licenses/BSD-3-Clause',
    'BSD 3-Clause': 'https://opensource.org/licenses/BSD-3-Clause',
    'CC-BY-4.0': 'https://creativecommons.org/licenses/by/4.0/',
}

## 3. GitHub API Functions

In [47]:
def compute_hash(text_or_bytes):
    """
    Computes MD5 hash for deduplication.
    
    Args:
        text_or_bytes: String or bytes to hash
        
    Returns:
        str: MD5 hash
    """
    if isinstance(text_or_bytes, str):
        return hashlib.md5(text_or_bytes.encode('utf-8', errors='ignore')).hexdigest()
    return hashlib.md5(text_or_bytes).hexdigest()

def extract_github_metadata(repo_url: str) -> Optional[Dict]:
    """Extract metadata from GitHub repository using API.
    This version fetches all README-like files in the repository root (case-insensitive)
    and merges their unique contents (deduplicated by MD5) into `readme_content`."""
    try:
        # Parse owner/repo from URL
        match = re.match(r'https?://github\.com/([^/]+)/([^/]+)', repo_url)
        if not match:
            logger.error("Invalid GitHub repository URL.")
            return None
        
        owner, repo = match.groups()
        api_url = f'https://api.github.com/repos/{owner}/{repo}'
        
        # Get repository info
        response = requests.get(api_url, headers=HEADERS)
        if response.status_code != 200:
            logger.error(f"GitHub API error: {response.status_code}")
            return None
        
        repo_data = response.json()
        
        # Get all README-like files from repository root
        readme_content = None
        try:
            contents_url = f'{api_url}/contents/'
            contents_resp = requests.get(contents_url, headers=HEADERS)
            unique_hashes = set()
            unique_contents = []
            if contents_resp.status_code == 200:
                repo_contents = contents_resp.json()
                for item in repo_contents:
                    if item.get('type') == 'file':
                        lower_name = item.get('name', '').lower()
                        if lower_name.startswith('readme'):
                            # Use the API content URL provided by the item
                            file_api_url = item.get('url') or f'https://api.github.com/repos/{owner}/{repo}/contents/{item.get(
)}'
                            file_resp = requests.get(file_api_url, headers=HEADERS)
                            if file_resp.status_code == 200:
                                file_data = file_resp.json()
                                if file_data.get('content'):
                                    content = base64.b64decode(file_data['content']).decode('utf-8')
                                    h = compute_hash(content)
                                    if h not in unique_hashes:
                                        unique_hashes.add(h)
                                        unique_contents.append(content)
        except Exception as e:
            logger.warning(f"Error fetching README files: {e}")
        
        # Merge unique README contents
        if unique_contents:
            # Join with double newline to preserve separation
            readme_content = '\n\n'.join(unique_contents)
        
        # Get programming languages
        languages = []
        languages_api_url = f'{api_url}/languages'
        languages_response = requests.get(languages_api_url, headers=HEADERS)
        if languages_response.status_code == 200:
            languages_data = languages_response.json()
            languages = list(languages_data.keys())
        
        # Extract structured metadata files
        structured_files = extract_structured_files(owner, repo)
        
        # Build metadata
        metadata = {
            'codeRepository': repo_data.get('html_url'),
            'name': repo_data.get('name'),
            'description': repo_data.get('description'),
            'programmingLanguage': repo_data.get('language'),
            'programmingLanguages': languages,
            'author': repo_data.get('owner', {}).get('login'),
            'dateCreated': repo_data.get('created_at'),
            'dateModified': repo_data.get('updated_at'),
            'keywords': repo_data.get('topics', []),
            'license': repo_data.get('license', {}).get('spdx_id') if repo_data.get('license') else None,
            'issueTracker': repo_data.get('issues_url', '').replace('{/number}', ''),
            'downloadUrl': repo_data.get('archive_url', '').replace('{archive_format}{/ref}', 'zipball/main'),
            'readme_content': readme_content,
            'structured_files': structured_files
        }
        
        return metadata
    except Exception as e:
        logger.error(f"Error extracting GitHub metadata: {e}")
        return None

def extract_structured_files(owner: str, repo: str) -> Dict[str, str]:
    """Extract structured metadata files from repository."""
    structured_files = {}
    
    # Define metadata files to extract
    metadata_files = {
        'citation.cff': 'CITATION.cff',
        'codemeta.json': 'codemeta.json',
        '.zenodo.json': '.zenodo.json',
        'package.json': 'package.json',
        'pyproject.toml': 'pyproject.toml',
        'setup.py': 'setup.py',
        'cargo.toml': 'Cargo.toml',
        'pom.xml': 'pom.xml',
        'composer.json': 'composer.json',
    }
    
    try:
        # Get repository contents
        contents_url = f'https://api.github.com/repos/{owner}/{repo}/contents/'
        response = requests.get(contents_url, headers=HEADERS)
        
        if response.status_code == 200:
            repo_contents = response.json()
            file_lookup = {item['name'].lower(): item['name'] for item in repo_contents if item['type'] == 'file'}
            
            for file_key, preferred_name in metadata_files.items():
                if file_key in file_lookup:
                    actual_filename = file_lookup[file_key]
                    file_url = f'https://api.github.com/repos/{owner}/{repo}/contents/{actual_filename}'
                    file_response = requests.get(file_url, headers=HEADERS)
                    
                    if file_response.status_code == 200:
                        file_data = file_response.json()
                        if file_data.get('content'):
                            content = base64.b64decode(file_data['content']).decode('utf-8')
                            structured_files[actual_filename] = content
    
    except Exception as e:
        logger.warning(f"Error extracting structured files: {e}")
    
    return structured_files

## 4. README Preprocessing Functions

In [48]:
def clean_readme_text(text: str, software_name: str = None) -> str:
    """Clean README text while preserving structure and readability.
    
    Args:
        text: README text to clean
        software_name: Name of the software to mask in installation commands
    """
    if not text or not text.strip():
        return ""
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove images but keep badges
    text = re.sub(r'!\[[^\]]*\]\([^)]+\)', '', text)
    
    # Clean markdown links - extract text and URL
    text = re.sub(r'\[([^\]]+)\]\(([^)]+)\)', r'[\1] \2', text)
    
    # Remove code fence markers but keep the content
    text = re.sub(r'^```\s*\w*\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^~~~\s*\w*\s*$', '', text, flags=re.MULTILINE)
    
    # Remove inline code backticks but keep content
    text = re.sub(r'`([^`]+)`', r'\1', text)
    
    # Mask installation commands that mention the software name
    if software_name:
        # Case-insensitive matching for pip install, conda install, etc.
        install_patterns = [
            rf'\bpip\s+install\s+{re.escape(software_name)}\b',
            rf'\bconda\s+install\s+{re.escape(software_name)}\b',
            rf'\bnpm\s+install\s+{re.escape(software_name)}\b',
            rf'\byarn\s+add\s+{re.escape(software_name)}\b',
            rf'\bgem\s+install\s+{re.escape(software_name)}\b',
            rf'\binstall\.packages\s*\(\s*["\']?{re.escape(software_name)}["\']?\s*\)',  # R
        ]
        
        for pattern in install_patterns:
            text = re.sub(pattern, 'XXX_SOFTWARE_INSTALLATION', text, flags=re.IGNORECASE)
    
    # Clean markdown formatting
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*]+)\*', r'\1', text)
    text = re.sub(r'^[ \t]*[=\-_*]{3,}[ \t]*$', '', text, flags=re.MULTILINE)
    
    # Protect list items: add double newline after each list item to preserve them
    text = re.sub(r'^[ \t]*[-*+][ \t]+(.+)$', r'\1\n', text, flags=re.MULTILINE)
    text = re.sub(r'^[ \t]*\d+\.[ \t]+(.+)$', r'\1\n', text, flags=re.MULTILINE)
    
    # Clean blockquotes
    text = re.sub(r'^[ \t]*>+[ \t]*', '', text, flags=re.MULTILINE)
    
    # Normalize excessive whitespace
    text = re.sub(r'[ \t]+', ' ', text)  # Multiple spaces/tabs to single space
    
    # Protect headers: add an extra newline after headers so they don't get merged
    text = re.sub(r'(^#{1,6}\s+.+)$', r'\1\n', text, flags=re.MULTILINE)
    
    # Replace single newlines with spaces (merge lines into paragraphs)
    # But keep intentional paragraph breaks (double newlines), headers, and list items
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    
    # Normalize multiple newlines to double newlines (paragraph breaks)
    text = re.sub(r'\n{2,}', '\n\n', text)
    
    # Clean up extra spaces around newlines
    text = re.sub(r' *\n *', '\n', text)
    
    # Remove empty lines that only contain whitespace
    text = re.sub(r'\n\s*\n', '\n\n', text)
    
    return text.strip()

In [49]:
def parse_readme_to_sections(md_text: str, max_tokens: int = MAX_TOKENS):
    """
    Parses Markdown text intelligently:
    1. If entire text fits in token limit, return as single section
    2. If too large, split by headers and merge sections to create balanced chunks
    3. Only split individual sections if they exceed the limit on their own
    
    Args:
        md_text (str): Markdown text
        
    Returns:
        list: List of sections with title and content
    """
    
    if not md_text.strip():
        return []
    
    # Check if entire text fits within limit
    estimated_tokens = len(md_text) / 4  # 1 token ≈ 4 chars
    
    if estimated_tokens <= max_tokens:
        # Return entire README as single section
        return [{
            "section": "README",
            "content": md_text.strip()
        }]
    
    # Text is too large, split by headers first
    sections = _split_by_headers(md_text)
    
    # Merge sections together to create balanced chunks
    merged_chunks = _merge_sections_into_chunks(sections, max_tokens)
    
    return merged_chunks


def _split_by_headers(md_text):
    """
    Splits markdown text by top-level headers (# only, not ## or ###).
    Preserves all newlines and spacing in content.
    
    Args:
        md_text (str): Markdown text
        
    Returns:
        list: List of sections with title and content
    """
    import re
    
    sections = []
    current_section = "Introduction"
    current_content = []
    found_any_header = False
    
    lines = md_text.split('\n')
    
    for line in lines:
        # Check if line is a TOP-LEVEL header (single # only)
        header_match = re.match(r'^#{1}\s+(.+)$', line)
        if header_match:
            found_any_header = True
            # Save previous section (only if it has content beyond the title)
            if current_content:
                content = '\n'.join(current_content).strip()
                if content and content != current_section:  # Don't save if only title exists
                    sections.append({
                        "section": current_section,
                        "content": content
                    })
            # Start new section with header as title
            current_section = header_match.group(1).strip()
            # Include section title in content (without the # symbol)
            current_content = [current_section]
        else:
            current_content.append(line)
    
    # Save last section (only if it has content)
    if current_content:
        content = '\n'.join(current_content).strip()
        if content:  # Only add if there's actual content
            sections.append({
                "section": current_section,
                "content": content
            })
    
    # If no sections with content found, return entire text as one section
    if not sections:
        sections.append({
            "section": "Content",
            "content": md_text.strip()
        })
    
    return sections


def _merge_sections_into_chunks(sections, max_tokens):
    """
    Merges multiple sections together into balanced chunks.
    Tries to fit as many sections as possible into each chunk without exceeding limit.
    
    Args:
        sections (list): List of section dictionaries
        max_tokens (int): Maximum tokens per chunk
        
    Returns:
        list: List of merged chunks
    """
    max_chars = max_tokens * 4  # Conservative estimate: 1 token ≈ 4 chars
    chunks = []
    
    current_chunk_sections = []
    current_chunk_length = 0
    
    for section in sections:
        section_length = len(section["content"])
        
        # If this single section exceeds the limit, handle it separately
        if section_length > max_chars:
            # Save current chunk if it has content
            if current_chunk_sections:
                chunks.append(_create_merged_chunk(current_chunk_sections))
                current_chunk_sections = []
                current_chunk_length = 0
            
            # Split the large section into multiple chunks
            split_chunks = _split_section_into_chunks(
                section["content"],
                section["section"],
                max_tokens
            )
            chunks.extend(split_chunks)
        
        # If adding this section would exceed limit, save current chunk and start new one
        elif current_chunk_length + section_length > max_chars and current_chunk_sections:
            chunks.append(_create_merged_chunk(current_chunk_sections))
            current_chunk_sections = [section]
            current_chunk_length = section_length
        
        # Add section to current chunk
        else:
            current_chunk_sections.append(section)
            current_chunk_length += section_length
    
    # Save last chunk
    if current_chunk_sections:
        chunks.append(_create_merged_chunk(current_chunk_sections))
    
    return chunks


def _create_merged_chunk(sections):
    """
    Creates a single chunk from multiple sections.
    
    Args:
        sections (list): List of section dictionaries to merge
        
    Returns:
        dict: Merged chunk with combined section names and content
    """
    if len(sections) == 1:
        return sections[0]
    
    # Combine section names
    section_names = [s["section"] for s in sections]
    combined_name = " + ".join(section_names[:3])  # Limit to first 3 names
    if len(section_names) > 3:
        combined_name += f" (and {len(section_names) - 3} more)"
    
    # Combine content with section separators
    combined_content = "\n\n".join([s["content"] for s in sections])
    
    return {
        "section": combined_name,
        "content": combined_content
    }

def _split_section_into_chunks(content, section_name, max_tokens):
    """
    Splits a large section into smaller chunks that fit within token limit.
    Preserves complete subsections - never splits in the middle of a subsection.
    
    Args:
        content (str): Section content to split
        section_name (str): Name of the section
        max_tokens (int): Maximum tokens per chunk
        
    Returns:
        list: List of section chunks
    """
    import re
    
    max_chars = max_tokens * 4  # Conservative estimate: 1 token ≈ 4 chars
    chunks = []
    chunk_index = 1
    
    # First, try to identify subsections (## ### etc.)
    subsections = []
    current_subsection = {"header": None, "content": []}
    
    lines = content.split('\n')
    
    for line in lines:
        # Check if this is a subsection header (##, ###, etc.)
        subsection_match = re.match(r'^(#{2,6})\s+(.+)$', line)
        if subsection_match:
            # Save previous subsection if it has content
            if current_subsection["content"] or current_subsection["header"]:
                subsections.append(current_subsection)
            # Start new subsection
            current_subsection = {
                "header": line,
                "content": []
            }
        else:
            current_subsection["content"].append(line)
    
    # Save last subsection
    if current_subsection["content"] or current_subsection["header"]:
        subsections.append(current_subsection)
    
    # If no subsections found, treat entire content as one subsection
    if not subsections or (len(subsections) == 1 and not subsections[0]["header"]):
        # No subsections, fall back to paragraph-based splitting
        return _split_by_paragraphs(content, section_name, max_tokens)
    
    # Now group subsections into chunks, keeping complete subsections together
    current_chunk = []
    current_length = 0
    
    for subsection in subsections:
        # Build subsection text
        subsection_parts = []
        if subsection["header"]:
            subsection_parts.append(subsection["header"])
        subsection_parts.extend(subsection["content"])
        subsection_text = '\n'.join(subsection_parts)
        subsection_length = len(subsection_text)
        
        # If single subsection exceeds limit, we need to split it
        if subsection_length > max_chars:
            # Save current chunk if it has content
            if current_chunk:
                chunk_content = '\n'.join(current_chunk)
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": chunk_content
                })
                chunk_index += 1
                current_chunk = []
                current_length = 0
            
            # Split this large subsection by paragraphs
            subsection_chunks = _split_by_paragraphs(
                subsection_text,
                section_name,
                max_tokens
            )
            
            # Add subsection chunks with proper indexing
            for sub_chunk in subsection_chunks:
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": sub_chunk["content"]
                })
                chunk_index += 1
        
        # If adding this subsection would exceed limit, save current chunk
        elif current_length + subsection_length + 1 > max_chars and current_chunk:
            chunk_content = '\n'.join(current_chunk)
            chunks.append({
                "section": f"{section_name} (part {chunk_index})",
                "content": chunk_content
            })
            chunk_index += 1
            current_chunk = [subsection_text]
            current_length = subsection_length
        
        # Add subsection to current chunk
        else:
            current_chunk.append(subsection_text)
            current_length += subsection_length + 1  # +1 for newline
    
    # Save last chunk
    if current_chunk:
        chunk_content = '\n'.join(current_chunk)
        section_label = f"{section_name} (part {chunk_index})" if chunk_index > 1 else section_name
        chunks.append({
            "section": section_label,
            "content": chunk_content
        })
    
    return chunks


def _split_by_paragraphs(content, section_name, max_tokens):
    """
    Fallback method to split content by paragraphs when no subsections exist.
    
    Args:
        content (str): Content to split
        section_name (str): Name of the section
        max_tokens (int): Maximum tokens per chunk
        
    Returns:
        list: List of section chunks
    """
    import re
    
    max_chars = max_tokens * 4
    chunks = []
    chunk_index = 1
    
    # Split by double newlines (paragraphs)
    paragraphs = content.split('\n\n')
    
    current_chunk = []
    current_length = 0
    
    for para in paragraphs:
        para_length = len(para)
        
        # If single paragraph is too large, split by lines
        if para_length > max_chars:
            # Save current chunk
            if current_chunk:
                chunk_content = '\n\n'.join(current_chunk)
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": chunk_content
                })
                chunk_index += 1
                current_chunk = []
                current_length = 0
            
            # Split paragraph by lines
            lines = para.split('\n')
            temp_lines = []
            temp_length = 0
            
            for line in lines:
                line_length = len(line)
                
                if line_length > max_chars:
                    # Save accumulated lines
                    if temp_lines:
                        chunks.append({
                            "section": f"{section_name} (part {chunk_index})",
                            "content": '\n'.join(temp_lines)
                        })
                        chunk_index += 1
                        temp_lines = []
                        temp_length = 0
                    
                    # Split very long line by sentences
                    sentences = re.split(r'(?<=[.!?])\s+', line)
                    for sentence in sentences:
                        if len(sentence) > max_chars:
                            # Character-level split as last resort
                            for i in range(0, len(sentence), max_chars):
                                chunks.append({
                                    "section": f"{section_name} (part {chunk_index})",
                                    "content": sentence[i:i + max_chars]
                                })
                                chunk_index += 1
                        else:
                            chunks.append({
                                "section": f"{section_name} (part {chunk_index})",
                                "content": sentence
                            })
                            chunk_index += 1
                
                elif temp_length + line_length + 1 > max_chars:
                    if temp_lines:
                        chunks.append({
                            "section": f"{section_name} (part {chunk_index})",
                            "content": '\n'.join(temp_lines)
                        })
                        chunk_index += 1
                    temp_lines = [line]
                    temp_length = line_length
                else:
                    temp_lines.append(line)
                    temp_length += line_length + 1
            
            # Save remaining lines
            if temp_lines:
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": '\n'.join(temp_lines)
                })
                chunk_index += 1
        
        # If adding paragraph exceeds limit, save current chunk
        elif current_length + para_length + 2 > max_chars:
            if current_chunk:
                chunk_content = '\n\n'.join(current_chunk)
                chunks.append({
                    "section": f"{section_name} (part {chunk_index})",
                    "content": chunk_content
                })
                chunk_index += 1
            current_chunk = [para]
            current_length = para_length
        
        # Add paragraph to current chunk
        else:
            current_chunk.append(para)
            current_length += para_length + 2
    
    # Save last chunk
    if current_chunk:
        chunk_content = '\n\n'.join(current_chunk)
        section_label = f"{section_name} (part {chunk_index})" if chunk_index > 1 else section_name
        chunks.append({
            "section": section_label,
            "content": chunk_content
        })
    
    return chunks

## 5. NER Model Pipeline

In [50]:
class NERPipeline:
    def __init__(self, model_dir: str, threshold: float = NER_THRESHOLD):
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForTokenClassification.from_pretrained(model_dir).to(DEVICE)
        self.model.eval()
        self.id2label = self.model.config.id2label
        self.threshold = threshold
        self.allowed_gap_chars = set(" \t\n\r.,;:!?'\"-_()/[]{}")
        self.allowed_gap_words = {
            "the", "of", "in", "on", "for", "with", "by",
            "a", "an", "and", "to"
        }

    @torch.no_grad()
    def __call__(self, text: str):
        if not text:
            return []

        enc = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_TOKENS,
            return_offsets_mapping=True,
            padding=True
        )

        offsets = enc.pop("offset_mapping")[0]
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        outputs = self.model(**enc)
        probs = F.softmax(outputs.logits, dim=-1)

        label_ids = torch.argmax(probs, dim=-1)[0]
        confs = torch.max(probs, dim=-1)[0][0]

        entities = []
        current = None

        for (label_id, conf, (start, end)) in zip(label_ids, confs, offsets):

            # Skip special tokens (CLS, SEP, PAD)
            if start == 0 and end == 0:
                continue

            raw = self.id2label[label_id.item()]
            conf = conf.item()

            # --- Decode BIO ---
            if raw.startswith("B-"):
                bio = "B"
                typ = raw[2:]
            elif raw.startswith("I-"):
                bio = "I"
                typ = raw[2:]
            else:
                bio = "O"
                typ = None

            # --- Threshold: ONLY suppress low-confidence B-tags ---
            if bio == "B" and conf < self.threshold:
                bio = "O"
                typ = None

            # --- Repair orphan I at the very start ---
            if bio == "I" and current is None:
                bio = "B"

            # --- Handle O ---
            if bio == "O":
                if current:
                    gap_text = text[start:end]
                    gap_lower = gap_text.lower()

                    # 1) If the O-token is punctuation / whitespace → keep entity open
                    if gap_text and all(c in self.allowed_gap_chars for c in gap_text):
                        continue

                    # 2) If the O-token is a common stopword → keep entity open
                    if gap_lower in self.allowed_gap_words:
                        continue

                    # Otherwise: real boundary → close entity
                    entities.append(current)
                    current = None

                continue

            # --- BEGIN NEW ENTITY ---
            if bio == "B" or current is None:
                if current:
                    entities.append(current)
                current = {
                    "start": start.item(),
                    "end": end.item(),
                    "label": typ,
                    "confidence": conf,
                    "text": text[start:end]
                }
                continue

            # --- CONTINUE CURRENT ENTITY (I of same type) ---
            if bio == "I" and current["label"] == typ:
                current["end"] = end.item()
                current["text"] = text[current["start"]:end.item()]
                current["confidence"] = (current["confidence"] + conf) / 2
                continue

            # --- Type mismatch (rare) → close and start new ---
            if current:
                entities.append(current)

            current = {
                "start": start.item(),
                "end": end.item(),
                "label": typ,
                "confidence": conf,
                "text": text[start:end]
            }

        # Flush last entity
        if current:
            entities.append(current)

        return entities

def extract_metadata_from_readme(readme_content: str, software_name: str = None) -> Dict[str, Any]:
    """Extract metadata from README content using NER model."""
    if not readme_content:
        return {}
    
    # Initialize NER pipeline
    try:
        ner_pipeline = NERPipeline(MODEL_DIR)
    except Exception as e:
        logger.error(f"Failed to load NER model: {e}")
        return {}
    
    # Clean and chunk the README
    cleaned_text = clean_readme_text(readme_content, software_name)
    chunks = parse_readme_to_sections(cleaned_text)
    
    # Extract entities from all chunks
    all_entities = []
    for chunk in chunks:
        entities = ner_pipeline(chunk['content'])
        all_entities.extend(entities)
    
    logger.info(f"Total entities extracted: {len(all_entities)}")
    
    # Group entities by label
    metadata = {}
    entity_sequence = []
    
    for entity in all_entities:
        label = entity['label']
        text = entity['text'].strip()
        
        if not text:
            continue
        
        if label not in metadata:
            metadata[label] = []
        
        metadata[label].append(text)
        entity_sequence.append((label, text))
    
    metadata['_entity_sequence'] = entity_sequence
    
    return metadata

## 6. Structured Files Metadata Extraction

In [7]:

def extract_from_citation_cff(cff_content: str) -> Dict[str, Any]:
    """
    Parse CITATION.cff content from a string.
    
    Args:
        cff_content: The raw content of CITATION.cff as a string
        
    Returns:
        Structured metadata dictionary
    """
    logger.info("Starting CFF parsing...")
    
    # Step 1: Preprocess
    cff_content = preprocess_content(cff_content)
    # Step 2: Fix structure
    cff_content = fix_structure(cff_content)
    # Step 3: Parse YAML
    raw_data = parse_yaml(cff_content)
    # Step 4: Transform to metadata
    metadata = transform_to_metadata(raw_data)
    
    return metadata


def preprocess_content(content: str) -> str:
    """Remove BOM, fix line endings, remove null bytes."""
    if content.startswith('\ufeff'):
        content = content[1:]
        logger.debug("Removed BOM")
    
    content = content.replace('\r\n', '\n').replace('\r', '\n')
    content = content.replace('\x00', '')
    content = content.strip()
    
    return content


def fix_structure(content: str) -> str:
    """
    Fix structural YAML issues by inserting missing section headers.
    Only inserts headers that are actually missing - once per section.
    """
    lines = content.split('\n')
    fixed_lines = []
    last_section_header = None
    
    for i, line in enumerate(lines):
        stripped = line.strip()
        
        # Skip empty and comment lines
        if not stripped or stripped.startswith('#'):
            fixed_lines.append(line)
            continue
        
        # Check if this is the start of an author list
        if stripped.startswith('- family-names:') or stripped.startswith('- given-names:'):
            # Check if previous non-empty line has 'authors:' or we just added it
            has_authors_header = False
            for prev_line in reversed(fixed_lines):
                if prev_line.strip() and not prev_line.strip().startswith('#'):
                    has_authors_header = prev_line.strip() == 'authors:' or prev_line.strip().startswith('- ')
                    break
            
            # Only insert if we don't have the header AND we haven't just added it
            if not has_authors_header and last_section_header != 'authors':
                logger.debug(f"Line {i}: Inserting 'authors:' before '{stripped[:30]}...'")
                fixed_lines.append('authors:')
                last_section_header = 'authors'
        
        # Check if this is the start of an identifier list
        elif stripped.startswith('- type:') and i > 0:
            # Check if this is under identifiers
            has_identifiers_header = False
            for prev_line in reversed(fixed_lines):
                if prev_line.strip() and not prev_line.strip().startswith('#'):
                    has_identifiers_header = prev_line.strip() == 'identifiers:' or prev_line.strip().startswith('- ')
                    break
            
            # Only insert if we don't have the header AND we haven't just added it
            if not has_identifiers_header and last_section_header != 'identifiers':
                logger.debug(f"Line {i}: Inserting 'identifiers:' before '{stripped[:30]}...'")
                fixed_lines.append('identifiers:')
                last_section_header = 'identifiers'
        
        # Track when we're moving to a new top-level section
        elif ':' in stripped and not stripped.startswith('-') and line[0] not in (' ', '\t'):
            last_section_header = None
        
        fixed_lines.append(line)
    
    return '\n'.join(fixed_lines)


def parse_yaml(content: str) -> Dict[str, Any]:
    """Parse YAML content with error handling."""
    try:
        data = yaml.safe_load(content)
        if data is None:
            data = {}
        logger.info("✓ Successfully parsed with YAML")
        return data
    except yaml.YAMLError as e:
        logger.error(f"✗ YAML parsing failed: {e}")
        logger.warning("Attempting fallback regex parsing...")
        return parse_yaml_fallback(content)


def parse_yaml_fallback(content: str) -> Dict[str, Any]:
    """Fallback regex-based parser when YAML fails."""
    data = {}
    lines = content.split('\n')
    current_author = {}
    authors = []
    
    for line in lines:
        stripped = line.strip()
        
        if not stripped or stripped.startswith('#'):
            continue
        
        # Author list item
        if stripped.startswith('- family-names:'):
            if current_author:
                authors.append(current_author)
            current_author = {}
            match = re.match(r'- family-names:\s*(.*)', stripped)
            if match:
                current_author['family-names'] = match.group(1).strip('"\'')
        
        # Author field
        elif current_author is not None and re.match(r'^\s+(given-names|email|affiliation|orcid):', line):
            match = re.match(r'\s+(\w+):\s*(.*)', line)
            if match:
                key = match.group(1)
                value = match.group(2).strip('"\'')
                current_author[key] = value
        
        # Top-level field
        elif re.match(r'^[a-z-]+:\s*', line):
            if current_author:
                authors.append(current_author)
                current_author = {}
            
            match = re.match(r'^([a-z-]+):\s*(.*)', line)
            if match:
                key = match.group(1)
                value = match.group(2).strip('"\'')
                data[key] = value
    
    if current_author:
        authors.append(current_author)
    
    if authors:
        data['authors'] = authors
    
    logger.info(f"✓ Fallback parser extracted {len(authors)} authors")
    return data


def transform_to_metadata(raw_data: Dict[str, Any]) -> Dict[str, Any]:
    """Transform raw CFF data to standardized metadata format."""
    metadata = {}
    
    # Basic field mappings
    field_mappings = {
        'title': 'name',
        'abstract': 'description',
        'version': 'version',
        'date-released': 'datePublished',
        'repository-code': 'codeRepository',
        'url': 'url',
        'license': 'license',
        'keywords': 'keywords'
    }
    
    for cff_key, meta_key in field_mappings.items():
        if cff_key in raw_data and raw_data[cff_key]:
            value = raw_data[cff_key]
            
            # Clean up multi-line strings
            if isinstance(value, str):
                value = ' '.join(value.split())
               # If still a date/datetime (nested structure), convert to ISO string
            try:
                if isinstance(value, (datetime.date, datetime)):
                    value = value.isoformat()
            except Exception:
                pass
            metadata[meta_key] = value
            logger.debug(f"  ✓ Mapped '{cff_key}' → '{meta_key}'")
    
    # Process authors
    authors = extract_authors(raw_data.get('authors', []))
    if authors:
        metadata['author'] = authors
        logger.info(f"✓ Extracted {len(authors)} authors")
    
    # Process identifiers
    identifiers = extract_identifiers(raw_data.get('identifiers', []))
    if identifiers:
        metadata['identifiers'] = identifiers
        logger.info(f"✓ Extracted {len(identifiers)} identifiers")
    
    return metadata


def extract_authors(authors_data: Any) -> List[Dict[str, Any]]:
    """Extract and normalize author information."""
    if not authors_data:
        return []
    
    if not isinstance(authors_data, list):
        authors_data = [authors_data]
    
    authors = []
    
    for idx, author in enumerate(authors_data):
        if not isinstance(author, dict):
            logger.warning(f"  ⚠ Skipping non-dict author #{idx}: {author}")
            continue
        
        logger.debug(f"  Author #{idx+1} raw data: {author}")
        
        author_obj = {'@type': 'Person'}
        
        # Extract given and family names
        given = author.get('given-names', '').strip()
        family = author.get('family-names', '').strip()
        
        logger.debug(f"    Given names: '{given}', Family names: '{family}'")
        
        # Construct full name
        if given and family:
            author_obj['name'] = f"{given} {family}"
        elif family:
            author_obj['name'] = family
        elif given:
            author_obj['name'] = given
        
        # Add separate givenName and familyName fields (for codemeta)
        if given:
            author_obj['givenName'] = given
        if family:
            author_obj['familyName'] = family
        
        # Add email
        if 'email' in author and author['email']:
            author_obj['email'] = author['email'].strip()
        
        # Add ORCID
        if 'orcid' in author and author['orcid']:
            orcid = author['orcid'].strip()
            if not orcid.startswith('http'):
                orcid = f"https://orcid.org/{orcid}"
            author_obj['@id'] = orcid
        
        # Add affiliation
        if 'affiliation' in author and author['affiliation']:
            author_obj['affiliation'] = {
                '@type': 'Organization',
                'name': author['affiliation'].strip()
            }
        
        # Only add if we have a name
        if 'name' in author_obj:
            authors.append(author_obj)
            logger.debug(f"  ✓ Author #{idx+1}: {author_obj['name']}")
    
    return authors


def extract_identifiers(identifiers_data: Any) -> List[Dict[str, Any]]:
    """Extract identifier information."""
    if not identifiers_data:
        return []
    
    if not isinstance(identifiers_data, list):
        identifiers_data = [identifiers_data]
    
    identifiers = []
    
    for identifier in identifiers_data:
        if isinstance(identifier, dict):
            identifiers.append(identifier)
    
    return identifiers


In [8]:

def extract_from_package_json(content: str) -> Dict[str, Any]:
    """Extract metadata from package.json file."""
    try:
        data = json.loads(content)
        
        metadata = {}
        
        # Basic fields
        if 'name' in data:
            metadata['name'] = data['name']
        if 'description' in data:
            metadata['description'] = data['description']
        if 'version' in data:
            metadata['version'] = data['version']
        if 'homepage' in data:
            metadata['url'] = data['homepage']
        if 'license' in data:
            metadata['license'] = data['license']
        if 'keywords' in data:
            metadata['keywords'] = data['keywords']
        
        # Repository
        if 'repository' in data:
            if isinstance(data['repository'], dict) and 'url' in data['repository']:
                metadata['codeRepository'] = data['repository']['url']
            elif isinstance(data['repository'], str):
                metadata['codeRepository'] = data['repository']
        
        # Author
        if 'author' in data:
            if isinstance(data['author'], str):
                metadata['author'] = [{'@type': 'Person', 'name': data['author']}]
            elif isinstance(data['author'], dict):
                author_obj = {'@type': 'Person'}
                if 'name' in data['author']:
                    author_obj['name'] = data['author']['name']
                if 'email' in data['author']:
                    author_obj['email'] = data['author']['email']
                metadata['author'] = [author_obj]
        
        return metadata
    except Exception as e:
        logger.warning(f"Error parsing package.json: {e}")
        return {}

def extract_from_pyproject_toml(content: str) -> Dict[str, Any]:
    """Extract metadata from pyproject.toml file."""
    try:
        import tomli
        data = tomli.loads(content)
        
        metadata = {}
        
        if 'project' in data:
            project = data['project']
            
            if 'name' in project:
                metadata['name'] = project['name']
            if 'description' in project:
                metadata['description'] = project['description']
            if 'version' in project:
                metadata['version'] = project['version']
            if 'license' in project:
                if isinstance(project['license'], dict) and 'text' in project['license']:
                    metadata['license'] = project['license']['text']
                elif isinstance(project['license'], str):
                    metadata['license'] = project['license']
            if 'keywords' in project:
                metadata['keywords'] = project['keywords']
            
            # URLs
            if 'urls' in project:
                urls = project['urls']
                if 'Homepage' in urls:
                    metadata['url'] = urls['Homepage']
                if 'Repository' in urls:
                    metadata['codeRepository'] = urls['Repository']
            
            # Authors
            if 'authors' in project:
                authors = []
                for author in project['authors']:
                    author_obj = {'@type': 'Person'}
                    if 'name' in author:
                        author_obj['name'] = author['name']
                    if 'email' in author:
                        author_obj['email'] = author['email']
                    authors.append(author_obj)
                metadata['author'] = authors
        
        return metadata
    except Exception as e:
        logger.warning(f"Error parsing pyproject.toml: {e}")
        return {}

def extract_from_structured_files(structured_files: Dict[str, str]) -> Dict[str, Any]:
    """Extract metadata from all structured files."""
    metadata = {}
    
    for filename, content in structured_files.items():
        file_metadata = {}
        
        if filename.lower() == 'citation.cff':
            file_metadata = extract_from_citation_cff(content)
        elif filename.lower() == 'package.json':
            file_metadata = extract_from_package_json(content)
        elif filename.lower() == 'pyproject.toml':
            file_metadata = extract_from_pyproject_toml(content)
        elif filename.lower() == 'codemeta.json':
            try:
                file_metadata = json.loads(content)
            except Exception as e:
                logger.warning(f"Error parsing codemeta.json: {e}")
        
        # Merge metadata with priority to first occurrence
        for key, value in file_metadata.items():
            if key not in metadata and value:
                metadata[key] = value
    
    return metadata

## 7. Priority-based Metadata Merging

In [51]:
def create_codemeta_base() -> Dict[str, str]:
    """Create base CodeMeta structure."""
    return {
        "@context": "https://w3id.org/codemeta/3.0",
        "@type": "SoftwareSourceCode"
    }

def normalize_author(author_data) -> Dict[str, Any]:
    """Normalize author data to a cleaned Person dict or Organization dict.

    Returns a dict with '@type' == 'Person' or '@type' == 'Organization'.
    Person entries include 'givenName' and 'familyName' (title-cased) when possible.
    Organization entries include 'name'.
    """
    # Helper to title-case names safely
    def _norm_name(s):
        try:
            return str(s).strip().title()
        except Exception:
            return str(s).strip()

    # If a simple string is provided, try to split into given/family
    if isinstance(author_data, str):
        s = author_data.strip()
        if not s:
            return {'@type': 'Person'}
        parts = s.split()
        result = {'@type': 'Person'}
        if len(parts) == 1:
            result['givenName'] = _norm_name(parts[0])
        else:
            result['givenName'] = _norm_name(' '.join(parts[:-1]))
            result['familyName'] = _norm_name(parts[-1])
        return result

    # If dict-like, first detect explicit Organization
    if isinstance(author_data, dict):
        # GitHub owner dicts often include 'type' key with value 'User' or 'Organization'
        a_type = author_data.get('type') or author_data.get('@type')
        if a_type and str(a_type).lower().startswith('org'):
            # Return an Organization representation
            name = author_data.get('name') or author_data.get('login') or author_data.get('company')
            return {'@type': 'Organization', 'name': str(name).strip() if name else ''}

        # Otherwise, treat as person-like
        result = {'@type': 'Person'}
        given = author_data.get('given-names') or author_data.get('givenName') or author_data.get('given')
        family = author_data.get('family-names') or author_data.get('familyName') or author_data.get('family')
        name_field = author_data.get('name')

        if given:
            result['givenName'] = _norm_name(given)
        if family:
            result['familyName'] = _norm_name(family)

        # If no given/family but a name exists, try splitting that
        if 'givenName' not in result and 'familyName' not in result and name_field:
            parts = str(name_field).strip().split()
            if len(parts) == 1:
                result['givenName'] = _norm_name(parts[0])
            else:
                result['givenName'] = _norm_name(' '.join(parts[:-1]))
                result['familyName'] = _norm_name(parts[-1])

        # Email
        email = author_data.get('email')
        if email:
            try:
                result['email'] = str(email).strip()
            except Exception:
                result['email'] = email

        # ORCID / id
        orcid = author_data.get('orcid') or author_data.get('@id') or author_data.get('id')
        if orcid:
            orcid = str(orcid).strip()
            if orcid and not orcid.startswith('http'):
                orcid = 'https://orcid.org/' + orcid
            result['@id'] = orcid

        # Affiliation: if object, extract name, else coerce to str
        affiliation = author_data.get('affiliation')
        if affiliation:
            if isinstance(affiliation, dict):
                aff_name = affiliation.get('name') or affiliation.get('organization') or ''
            else:
                aff_name = affiliation
            if aff_name:
                result['affiliation'] = str(aff_name).strip()

        return result

    # Fallback
    return {'@type': 'Person'}


def merge_metadata_with_priority(github_metadata: Dict, structured_metadata: Dict, model_metadata: Dict) -> Dict[str, Any]:
    """Merge metadata with priority: GitHub > Structured Files > Model extraction.

    Authors are merged and deduplicated; Organization entries are not included
    in the final `author` list (we only include Person entries).
    """
    codemeta = create_codemeta_base()

    # Priority order: GitHub, Structured, Model
    sources = [
        ("github", github_metadata),
        ("structured", structured_metadata),
        ("model", model_metadata)
    ]

    # Scalar fields (first non-empty wins)
    scalar_fields = [
        "name", "description", "version", "license", "url", "codeRepository",
        "issueTracker", "downloadUrl", "dateCreated", "dateModified", "datePublished",
        "programmingLanguage"
    ]

    for field in scalar_fields:
        for source_name, metadata in sources:
            if field in metadata and metadata[field]:
                codemeta[field] = metadata[field]
                break

    # List fields (combine unique values)
    list_fields = ["keywords"]

    for field in list_fields:
        combined = []
        seen = set()

        for source_name, metadata in sources:
            if field in metadata and metadata[field]:
                values = metadata[field] if isinstance(metadata[field], list) else [metadata[field]]
                for value in values:
                    if value and str(value).lower() not in seen:
                        combined.append(value)
                        seen.add(str(value).lower())

        if combined:
            codemeta[field] = combined

    # Author field (special handling)
    authors = []
    seen_authors = set()

    for source_name, metadata in sources:
        if "author" in metadata and metadata["author"]:
            author_data = metadata["author"]

            # Normalize to list
            author_list = author_data if isinstance(author_data, list) else [author_data]

            for author in author_list:
                normalized = normalize_author(author)

                # Only include Person entries
                if normalized.get('@type') != 'Person':
                    continue

                # Build dedupe key
                dedupe_key = None
                if normalized.get('@id'):
                    dedupe_key = str(normalized.get('@id')).lower()
                else:
                    gn = normalized.get('givenName', '')
                    fn = normalized.get('familyName', '')
                    if gn or fn:
                        dedupe_key = f"{gn.strip().lower()}|{fn.strip().lower()}"
                    elif normalized.get('email'):
                        dedupe_key = str(normalized.get('email')).lower()

                if not dedupe_key:
                    # Skip entries we can't reasonably deduplicate
                    continue

                if dedupe_key not in seen_authors:
                    # Ensure minimal structure and avoid adding a merged 'name'
                    author_obj = {'@type': 'Person'}
                    if normalized.get('@id'):
                        author_obj['@id'] = normalized.get('@id')
                    if normalized.get('email'):
                        author_obj['email'] = normalized.get('email')
                    if normalized.get('givenName'):
                        author_obj['givenName'] = normalized.get('givenName')
                    if normalized.get('familyName'):
                        author_obj['familyName'] = normalized.get('familyName')
                    if normalized.get('affiliation'):
                        author_obj['affiliation'] = normalized.get('affiliation')

                    authors.append(author_obj)
                    seen_authors.add(dedupe_key)

    if authors:
        codemeta["author"] = authors

    # Add model-extracted entities as additional fields
    if model_metadata:
        model_fields = {}
        for label, entities in model_metadata.items():
            if entities and label != "O":  # Skip "O" (outside) labels
                field_name = f"extracted_{label.lower()}"
                model_fields[field_name] = entities

        if model_fields:
            codemeta["_modelExtracted"] = model_fields

    # Add source information
    codemeta["_sources"] = {
        "github": bool(github_metadata),
        "structured": bool(structured_metadata),
        "model": bool(model_metadata)
    }

    return codemeta

In [52]:

def preprocess_model_metadata(raw_model_meta: Dict[str, List[str]], existing_codemeta: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Preprocess NER model output to match ground truth CodeMeta format.
    
    All returned fields match the Label Studio ground truth format exactly:
    - Single items are returned as single values (string or dict)
    - Multiple items are returned as lists
    - None if no items found for that field
    
    Returns:
        Dict with fields: license, citation, buildInstructions, operatingSystem,
                         runtimePlatform, softwareRequirements, related_link
    """
    if not raw_model_meta:
        return existing_codemeta or {}
    
    result = existing_codemeta.copy() if existing_codemeta else {}
    
    # Filter out I- prefixed (incomplete) entities before processing
    merged_entities = _filter_incomplete_entities(raw_model_meta)
    
    logger.info(f"Raw model metadata before preprocess: {raw_model_meta}")
    logger.info(f"After filtering incomplete entities: {merged_entities}")
    
    # Process all entity types
    # Each function returns: single item, list of items, or None
    licenses = _process_licenses(merged_entities, result.get('license'))
    if licenses is not None:
        result['license'] = licenses
    
    citations = _process_citations(merged_entities)
    if citations is not None:
        result['citation'] = citations
    
    builds = _process_build_instructions(merged_entities)
    if builds is not None:
        result['buildInstructions'] = builds
    
    os_systems = _process_operating_systems(merged_entities)
    if os_systems is not None:
        result['operatingSystem'] = os_systems
    
    runtimes = _process_runtime_platforms(merged_entities)
    if runtimes is not None:
        result['runtimePlatform'] = runtimes
    
    software_reqs = _process_software_requirements(merged_entities)
    if software_reqs is not None:
        result['softwareRequirements'] = software_reqs
    
    related = _process_related_links(merged_entities)
    if related is not None:
        result['relatedLink'] = related
    
    # Process reference publications (if present)
    references = _process_reference_publications(merged_entities)
    if references is not None:
        result['referencePublication'] = references
    
    # Process continuous integration (if present)
    ci = _process_continuous_integration(merged_entities)
    if ci is not None:
        result['continuousIntegration'] = ci
    
    return result


def _filter_incomplete_entities(raw_model_meta: Dict[str, List[str]]) -> Dict[str, List[str]]:
    """Filter out I- prefixed (incomplete) entities and clean B- prefixes.
    
    Now properly handles entity sequences without over-merging.
    """
    # First, merge consecutive entities using the sequence (only merges I- tags)
    if '_entity_sequence' in raw_model_meta:
        merged = _merge_consecutive_entities(raw_model_meta['_entity_sequence'])
        raw_model_meta = merged
    
    filtered = {}
    
    # Minimum lengths for different entity types
    MIN_LENGTHS = {
        'LICENSE': 3,
        'OPERATING_SYSTEM': 3,
        'CITATION': 10,  # Citations should be substantial
        'CITATION_BIBTEX': 30,  # BibTeX entries should be complete
        'REFERENCE_PUBLICATION': 15,  # References should be substantial
        'REFERENCE_PUBLICATION_BIBTEX': 30,
        'SOFTWARE_REQUIREMENTS': 2,
        'default': 2
    }
    
    for key, values in raw_model_meta.items():
        # Skip I- prefixed labels (should already be filtered by merge)
        if key.startswith('I-'):
            continue
        
        # Skip special keys
        if key == '_entity_sequence':
            continue
        
        # Remove B- prefix if present
        clean_key = key[2:] if key.startswith('B-') else key
        
        if clean_key not in filtered:
            filtered[clean_key] = []
        
        # Get minimum length for this entity type
        min_length = MIN_LENGTHS.get(clean_key, MIN_LENGTHS['default'])
        
        # Filter out short fragments
        for v in values:
            v_stripped = v.strip()
            
            if not v_stripped or len(v_stripped) < min_length:
                continue
            
            # Filter URL fragments
            if v_stripped in ['.org/', '.com/', '.net/', '.edu/', 'http://', 'https://']:
                continue
            
            # Filter incomplete license names
            if clean_key == 'LICENSE':
                incomplete_licenses = ['ap', 'lic', 'license', 'or']
                if v_stripped.lower() in incomplete_licenses:
                    continue
                if len(v_stripped) < 3 and v_stripped.upper() not in ['MIT', 'GPL', 'BSD', 'ISC', 'MPL', 'EPL']:
                    continue
            
            # Filter incomplete OS names
            if clean_key == 'OPERATING_SYSTEM':
                incomplete_os = ['ub', 'lin', 'win', 'mac', 'deb', 'fed']
                if v_stripped.lower() in incomplete_os:
                    continue
            
            # Filter citation/reference fragments
            if clean_key in ['CITATION', 'REFERENCE_PUBLICATION']:
                # Skip very short fragments (but be less aggressive than before)
                word_count = len(v_stripped.split())
                if word_count < 3:
                    continue
                
                # Skip common incomplete fragments
                fragment_patterns = [
                    r'^and others$',
                    r'^et al\.?$',
                    r'^\d{4}$',  # Just a year
                    r'^[A-Z]\.$',  # Just an initial
                    r'^[A-Z][a-z]+$',  # Single word like "Burgess"
                    r'^\w{1,3},?\s*[A-Z]\.?\s*$',  # "R." or "wijn, J"
                ]
                if any(re.match(pattern, v_stripped, re.IGNORECASE) for pattern in fragment_patterns):
                    continue
            
            filtered[clean_key].append(v_stripped)
    
    # Preserve filtered entity sequence
    if '_entity_sequence' in raw_model_meta:
        filtered_sequence = []
        for label, text in raw_model_meta['_entity_sequence']:
            if label.startswith('I-'):
                continue
            
            clean_label = label[2:] if label.startswith('B-') else label
            min_length = MIN_LENGTHS.get(clean_label, MIN_LENGTHS['default'])
            
            text_stripped = text.strip()
            
            if not text_stripped or len(text_stripped) < min_length:
                continue
            
            # Apply same filtering as above
            if text_stripped in ['.org/', '.com/', '.net/', '.edu/', 'http://', 'https://']:
                continue
            
            if clean_label == 'LICENSE':
                incomplete_licenses = ['ap', 'lic', 'license', 'or']
                if text_stripped.lower() in incomplete_licenses:
                    continue
                if len(text_stripped) < 3 and text_stripped.upper() not in ['MIT', 'GPL', 'BSD', 'ISC', 'MPL', 'EPL']:
                    continue
            
            if clean_label == 'OPERATING_SYSTEM':
                incomplete_os = ['ub', 'lin', 'win', 'mac', 'deb', 'fed']
                if text_stripped.lower() in incomplete_os:
                    continue
            
            if clean_label in ['CITATION', 'REFERENCE_PUBLICATION']:
                word_count = len(text_stripped.split())
                if word_count < 3:
                    continue
                
                fragment_patterns = [
                    r'^and others$',
                    r'^et al\.?$',
                    r'^\d{4}$',
                    r'^[A-Z]\.$',
                    r'^[A-Z][a-z]+$',
                    r'^\w{1,3},?\s*[A-Z]\.?\s*$',
                ]
                if any(re.match(pattern, text_stripped, re.IGNORECASE) for pattern in fragment_patterns):
                    continue
            
            filtered_sequence.append((clean_label, text_stripped))
        
        filtered['_entity_sequence'] = filtered_sequence
    
    return filtered

def _process_licenses(entities: Dict[str, List[str]], existing_license: Any) -> Any:
    """Process license information in ground truth format.
    
    Ground truth format:
    - Non-URL licenses: {"@type": "CreativeWork", "name": "license_name"}
    - URL licenses: "https://..."
    
    Returns single item or list matching ground truth.
    """
    LICENSE_URL_MAP = {
        'MIT': 'https://opensource.org/licenses/MIT',
        'Apache License, Version 2.0': 'https://www.apache.org/licenses/LICENSE-2.0',
        'Apache-2.0': 'https://www.apache.org/licenses/LICENSE-2.0',
        'Apache 2.0': 'https://www.apache.org/licenses/LICENSE-2.0',
        'GPL-3.0': 'https://www.gnu.org/licenses/gpl-3.0.html',
        'GPL-2.0': 'https://www.gnu.org/licenses/gpl-2.0.html',
        'BSD-3-Clause': 'https://opensource.org/licenses/BSD-3-Clause',
        'BSD-2-Clause': 'https://opensource.org/licenses/BSD-2-Clause',
    }
    
    license_names = entities.get('LICENSE', [])
    license_urls = entities.get('LICENSE_URL', [])
    
    all_licenses = []
    if existing_license:
        all_licenses.extend(existing_license if isinstance(existing_license, list) else [existing_license])
    
    # Process license names - format as {"@type": "CreativeWork", "name": "..."}
    for license_name in license_names:
        license_name = license_name.strip()
        
        # Skip invalid/incomplete names
        if len(license_name) < 3:
            continue
        
        # Check if we should convert to URL (optional - comment out to match ground truth exactly)
        # license_url = LICENSE_URL_MAP.get(license_name)
        # if license_url:
        #     all_licenses.append(license_url)
        # else:
        
        # Always format as CreativeWork object for non-URLs (matching ground truth)
        all_licenses.append({
            '@type': 'CreativeWork',
            'name': license_name
        })
    
    # Process license URLs - keep as plain strings
    for url in license_urls:
        cleaned_url = _clean_url(url)
        if cleaned_url and _is_valid_url(cleaned_url):
            all_licenses.append(cleaned_url)
    
    # Deduplicate while preserving order
    unique_licenses = []
    seen = set()
    for lic in all_licenses:
        # Create a hashable key for comparison
        if isinstance(lic, dict):
            key = lic.get('name', '')
        else:
            key = lic
        
        if key not in seen:
            seen.add(key)
            unique_licenses.append(lic)
    
    if not unique_licenses:
        return existing_license
    
    return unique_licenses[0] if len(unique_licenses) == 1 else unique_licenses


def _process_operating_systems(entities: Dict[str, List[str]]) -> Any:
    """Process operating system entities.
    
    Returns:
        - Single string if only one OS
        - List of strings if multiple
        - None if none found
    """
    os_systems = []
    
    os_names = entities.get('OPERATING_SYSTEM', [])
    
    for os_name in os_names:
        os_name = os_name.strip()
        
        # Skip short fragments (already filtered, but double-check)
        if len(os_name) < 3:
            continue
        
        # Normalize common OS names
        os_normalized = os_name
        if os_name.lower() == 'macos':
            os_normalized = 'macOS'
        elif os_name.lower() == 'linux':
            os_normalized = 'Linux'
        elif os_name.lower() == 'windows':
            os_normalized = 'Windows'
        elif os_name.lower() == 'ubuntu':
            os_normalized = 'Ubuntu'
        elif os_name.lower() == 'debian':
            os_normalized = 'Debian'
        
        os_systems.append(os_normalized)
    
    # Deduplicate (case-insensitive)
    unique_os = []
    seen_lower = set()
    for os in os_systems:
        if os.lower() not in seen_lower:
            unique_os.append(os)
            seen_lower.add(os.lower())
    
    if not unique_os:
        return None
    
    # Return format matching ground truth
    return unique_os[0] if len(unique_os) == 1 else unique_os

def _process_citations(entities: Dict[str, List[str]]) -> Any:
    """Process citation-related entities without over-merging."""
    citations = []
    
    # Get all citation data (each item is now a separate entity)
    citation_texts = entities.get('CITATION', [])
    citation_urls = entities.get('CITATION_URL', [])
    citation_bibtex = entities.get('CITATION_BIBTEX', [])
    
    # Process citation texts - each is a separate citation
    for citation_text in citation_texts:
        citation_text = citation_text.strip()
        
        # Skip very short fragments
        if len(citation_text) < 10:
            continue
        
        citation_obj = {
            '@type': 'CreativeWork',
            'text': citation_text
        }
        citations.append(citation_obj)
    
    # Process citation URLs - each is separate
    for url in citation_urls:
        cleaned_url = _clean_url(url)
        if cleaned_url and _is_valid_url(cleaned_url):
            citations.append(cleaned_url)
    
    # Process BibTeX entries - each is separate
    for bibtex in citation_bibtex:
        bibtex = bibtex.strip()
        
        if bibtex and len(bibtex) > 30:
            citation_obj = {
                '@type': 'CreativeWork',
                'encodingFormat': 'application/x-bibtex',
                'text': bibtex
            }
            citations.append(citation_obj)
    
    if not citations:
        return None
    
    return citations[0] if len(citations) == 1 else citations


def _process_build_instructions(entities: Dict[str, List[str]]) -> Any:
    """Merge documentation_url and build_instructions into build instructions array.
    
    Returns:
        - Single URL string if only one
        - List of URL strings if multiple
        - None if none found
    """
    build_instructions = []
    doc_urls = entities.get('DOCUMENTATION_URL', [])
    build_urls = entities.get('BUILD_INSTRUCTIONS', [])
    
    for url in doc_urls + build_urls:
        cleaned_url = _clean_url(url)
        if cleaned_url and _is_valid_url(cleaned_url):
            build_instructions.append(cleaned_url)
    
    # Deduplicate while preserving order
    unique_builds = list(dict.fromkeys(build_instructions))
    
    if not unique_builds:
        return None
    
    # Return format matching ground truth
    return unique_builds[0] if len(unique_builds) == 1 else unique_builds


def _process_runtime_platforms(entities: Dict[str, List[str]]) -> Any:
    """Process runtime platform entities.
    
    Returns:
        - Single string if only one platform
        - List of strings if multiple  
        - None if none found
    """
    platforms = []
    
    platform_names = entities.get('RUNTIME_PLATFORM', [])
    
    for platform in platform_names:
        platform = platform.strip()
        
        # Skip fragments
        if len(platform) < 2:
            continue
        
        platforms.append(platform)
    
    # Deduplicate (case-insensitive)
    unique_platforms = []
    seen_lower = set()
    for p in platforms:
        if p.lower() not in seen_lower:
            unique_platforms.append(p)
            seen_lower.add(p.lower())
    
    if not unique_platforms:
        return None
    
    # Return format matching ground truth
    return unique_platforms[0] if len(unique_platforms) == 1 else unique_platforms


def _process_software_requirements(entities: Dict[str, Any]) -> Any:
    """Process software requirements using entity sequence to maintain order.
    
    CRITICAL: Uses _entity_sequence to match requirements with their URLs/versions
    in the correct order as they appear in the document.
    
    Returns:
        - Single SoftwareApplication dict if only one
        - List of dicts if multiple
        - None if none found
    """
    requirements = []
    
    # Check if we have entity sequence (preferred for maintaining order)
    entity_sequence = entities.get('_entity_sequence', [])
    
    if entity_sequence:
        # Use sequence to maintain order and proper matching
        current_req = None
        
        for label, text in entity_sequence:
            text = text.strip()
            
            if label == 'SOFTWARE_REQUIREMENTS':
                # Skip very short names (likely NER errors)
                if len(text) < 2:
                    continue
                
                # Save previous requirement if exists
                if current_req:
                    requirements.append(current_req)
                
                # Start new requirement
                current_req = {
                    '@type': 'SoftwareApplication',
                    'name': text
                }
            
            elif label == 'SOFTWARE_REQUIREMENTS_URL' and current_req:
                url = _clean_url(text)
                if url and _is_valid_url(url):
                    current_req['url'] = url
            
            elif label == 'SOFTWARE_REQUIREMENTS_VERSION' and current_req:
                if text and len(text) > 0:
                    current_req['version'] = text
        
        # Don't forget the last requirement
        if current_req:
            requirements.append(current_req)
    
    else:
        # Fallback: use simple sequential matching (less accurate)
        req_names = entities.get('SOFTWARE_REQUIREMENTS', [])
        req_urls = entities.get('SOFTWARE_REQUIREMENTS_URL', [])
        req_versions = entities.get('SOFTWARE_REQUIREMENTS_VERSION', [])
        
        url_idx = 0
        version_idx = 0
        
        for name in req_names:
            cleaned_name = name.strip()
            
            if len(cleaned_name) < 2:
                continue
            
            req_obj = {
                '@type': 'SoftwareApplication',
                'name': cleaned_name
            }
            
            # Attach next available URL
            if url_idx < len(req_urls):
                url = _clean_url(req_urls[url_idx])
                if url and _is_valid_url(url):
                    req_obj['url'] = url
                url_idx += 1
            
            # Attach next available version
            if version_idx < len(req_versions):
                version = req_versions[version_idx].strip()
                if version:
                    req_obj['version'] = version
                version_idx += 1
            
            requirements.append(req_obj)
    
    # Apply smart deduplication
    deduplicated = _deduplicate_software_requirements(requirements)
    
    if not deduplicated:
        return None
    
    # Return format matching ground truth
    return deduplicated[0] if len(deduplicated) == 1 else deduplicated


def _process_related_links(entities: Dict[str, List[str]]) -> Any:
    """Process related links.
    
    Returns:
        - Single URL string if only one
        - List of URL strings if multiple
        - None if none found
    """
    links = []
    
    related_links = entities.get('RELATED_LINK', [])
    for link in related_links:
        link = link.strip()
        
        # Skip fragments
        if len(link) < 5:
            continue
        
        # Validate and clean URL
        cleaned_url = _clean_url(link)
        if cleaned_url and _is_valid_url(cleaned_url):
            links.append(cleaned_url)
    
    # Deduplicate
    unique_links = list(dict.fromkeys(links))
    
    if not unique_links:
        return None
    
    # Return format matching ground truth
    return unique_links[0] if len(unique_links) == 1 else unique_links

def _process_reference_publications(entities: Dict[str, List[str]]) -> Any:
    """Process reference publication entities without over-merging."""
    references = []
    
    # Get all reference data (each item is now a separate entity)
    ref_texts = entities.get('REFERENCE_PUBLICATION', [])
    ref_urls = entities.get('REFERENCE_PUBLICATION_URL', [])
    ref_bibtex = entities.get('REFERENCE_PUBLICATION_BIBTEX', [])
    
    # Process reference texts - each is a separate reference
    for ref_text in ref_texts:
        ref_text = ref_text.strip()
        
        # Skip very short fragments
        if len(ref_text) < 15:
            continue
        
        ref_obj = {
            '@type': 'ScholarlyArticle',
            'text': ref_text
        }
        references.append(ref_obj)
    
    # Process reference URLs - each is separate
    for url in ref_urls:
        cleaned_url = _clean_url(url)
        if cleaned_url and _is_valid_url(cleaned_url):
            ref_obj = {
                '@type': 'ScholarlyArticle',
                'url': cleaned_url
            }
            references.append(ref_obj)
    
    # Process BibTeX entries - each is separate
    for bibtex in ref_bibtex:
        bibtex = bibtex.strip()
        if bibtex and len(bibtex) > 30:
            ref_obj = {
                '@type': 'ScholarlyArticle',
                'encodingFormat': 'application/x-bibtex',
                'text': bibtex
            }
            references.append(ref_obj)
    
    if not references:
        return None
    
    return references[0] if len(references) == 1 else references



def _process_continuous_integration(entities: Dict[str, List[str]]) -> Any:
    """Process continuous integration URLs.
    
    Returns:
        - Single URL string if only one
        - List of URL strings if multiple
        - None if none found
    """
    ci_urls = []
    
    ci_entries = entities.get('CONTINUOUS_INTEGRATION', [])
    for ci_url in ci_entries:
        ci_url = ci_url.strip()
        
        # Skip fragments
        if len(ci_url) < 5:
            continue
        
        # Validate and clean URL
        cleaned_url = _clean_url(ci_url)
        if cleaned_url and _is_valid_url(cleaned_url):
            ci_urls.append(cleaned_url)
    
    # Deduplicate
    unique_ci_urls = list(dict.fromkeys(ci_urls))
    
    if not unique_ci_urls:
        return None
    
    # Return format matching ground truth
    return unique_ci_urls[0] if len(unique_ci_urls) == 1 else unique_ci_urls


def _deduplicate_software_requirements(reqs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Deduplicate software requirements intelligently.
    
    Rules:
    - Same name + same URL + same version: keep one
    - Same name + different URL: keep both
    - Same name + standalone vs with metadata: keep only with metadata
    - Same name + same URL + different version: keep most complete
    """
    if not reqs:
        return []
    
    seen = {}  # name -> list of entries
    
    for req in reqs:
        name = req.get('name', '').lower().strip()
        if not name:
            continue
        
        if name not in seen:
            seen[name] = []
        seen[name].append(req)
    
    result = []
    
    for name, entries in seen.items():
        if len(entries) == 1:
            result.append(entries[0])
            continue
        
        # Multiple entries with same name
        # Separate into entries with metadata and without
        with_metadata = [e for e in entries if e.get('url') or e.get('version')]
        without_metadata = [e for e in entries if not (e.get('url') or e.get('version'))]
        
        # If we have entries with metadata, discard standalone ones
        if with_metadata:
            # Deduplicate by name+url, keep the most complete entry
            unique_metadata = {}
            for entry in with_metadata:
                # Key: name + url (not version, so we compare versions)
                key = (entry.get('name', '').lower().strip(), entry.get('url', ''))
                
                if key not in unique_metadata:
                    unique_metadata[key] = entry
                else:
                    # Keep entry with more metadata (url + version > url only)
                    existing = unique_metadata[key]
                    existing_completeness = sum([bool(existing.get('url')), bool(existing.get('version'))])
                    new_completeness = sum([bool(entry.get('url')), bool(entry.get('version'))])
                    
                    if new_completeness > existing_completeness:
                        unique_metadata[key] = entry
            
            result.extend(unique_metadata.values())
        else:
            # All standalone, keep only first one
            result.append(entries[0])
    
    return result


def _clean_url(url: str) -> str:
    """Clean and normalize URL."""
    if not url:
        return ''
    
    url = url.strip()
    
    # Handle incomplete URLs from NER
    if '://' not in url:
        if url.startswith(('http', 'www')):
            url = re.sub(r'\s+', '', url)
        if url.startswith('www.'):
            url = 'https://' + url
    
    # Fix incomplete protocol
    if url.startswith('://'):
        url = 'https' + url
    
    return url


def _is_valid_url(url: str) -> bool:
    """Check if URL is valid and complete."""
    if not url or len(url) < 10:  # Minimum reasonable URL length
        return False
    
    url_pattern = re.compile(
        r'^https?://'
        r'(?:(?:[A-Z0-9](?:[A-Z0-9-]{0,61}[A-Z0-9])?\.)+[A-Z]{2,6}\.?|'
        r'localhost|'
        r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})'
        r'(?::\d+)?'
        r'(?:/?|[/?]\S+)$', re.IGNORECASE)
    
    return bool(url_pattern.match(url))

def _merge_consecutive_entities(entity_sequence: List[Tuple[str, str]]) -> Dict[str, List[str]]:
    """Merge consecutive tokens of the same entity type ONLY when using I- tags.
    
    CRITICAL CHANGE: Only merge when we see I- (inside) tags. B- (beginning) tags
    or plain labels start NEW entities, even if they're the same type.
    
    This prevents merging separate citations/references that happen to be adjacent.
    """
    merged = {}
    current_label = None
    current_tokens = []
    
    merged_sequence = []
    
    for label, text in entity_sequence:
        text = text.strip()
        if not text:
            continue
        
        # Determine if this is a continuation (I- prefix) or a new entity (B- or plain)
        if label.startswith('B-'):
            normalized_label = label[2:]
            is_continuation = False  # B- always starts new entity
        elif label.startswith('I-'):
            normalized_label = label[2:]
            is_continuation = True  # I- continues current entity
        else:
            normalized_label = label
            is_continuation = False  # Plain label starts new entity
        
        # CRITICAL: Only merge if it's an I- tag AND same type as current entity
        if is_continuation and normalized_label == current_label:
            # Continue current entity (I- tag)
            current_tokens.append(text)
        else:
            # Save previous entity if exists
            if current_label and current_tokens:
                merged_text = _merge_tokens(current_label, current_tokens)
                if merged_text:
                    if current_label not in merged:
                        merged[current_label] = []
                    merged[current_label].append(merged_text)
                    merged_sequence.append((current_label, merged_text))
            
            # Start new entity (B- tag or plain label)
            current_label = normalized_label
            current_tokens = [text]
    
    # Don't forget the last entity
    if current_label and current_tokens:
        merged_text = _merge_tokens(current_label, current_tokens)
        if merged_text:
            if current_label not in merged:
                merged[current_label] = []
            merged[current_label].append(merged_text)
            merged_sequence.append((current_label, merged_text))
    
    # Preserve merged sequence
    merged['_entity_sequence'] = merged_sequence
    
    return merged


def _merge_tokens(label: str, tokens: List[str]) -> str:
    """Merge tokens based on entity type."""
    if not tokens:
        return ''
    
    # Remove empty tokens
    tokens = [t for t in tokens if t.strip()]
    if not tokens:
        return ''
    
    # URL handling - concatenate without spaces
    if 'URL' in label or any(t in ['http', 'https', '://'] for t in tokens):
        merged = ''.join(tokens)
        merged = merged.replace('http ://', 'https://')
        merged = merged.replace('https ://', 'https://')
        merged = merged.replace('http://', 'https://')
        # Filter URL fragments
        if merged in ['.org/', '.com/', '.net/', '.edu/']:
            return ''
        return merged
    
    # BibTeX handling - preserve formatting
    if 'BIBTEX' in label:
        merged = ' '.join(tokens)
        
        # Clean up BibTeX formatting
        merged = re.sub(r'\s*,\s*', ', ', merged)
        merged = re.sub(r'\s*=\s*', ' = ', merged)
        merged = re.sub(r'\s*{\s*', '{', merged)
        merged = re.sub(r'\s*}\s*', '}', merged)
        merged = re.sub(r'\s+', ' ', merged)
        
        # Fix split braces
        merged = re.sub(r'{\s+', '{', merged)
        merged = re.sub(r'\s+}', '}', merged)
        
        return merged.strip()
    
    # Citation and Reference handling - join with spaces
    if label in ['CITATION', 'REFERENCE_PUBLICATION']:
        merged = ' '.join(tokens)
        
        # Clean up spacing
        merged = re.sub(r'\s+', ' ', merged)
        merged = re.sub(r'\s+([.,;:])', r'\1', merged)
        merged = re.sub(r'([.,;:])\s*([.,;:])', r'\1\2', merged)
        
        return merged.strip()
    
    # Default - join with spaces
    merged = ' '.join(tokens)
    merged = re.sub(r'\s+', ' ', merged)
    merged = re.sub(r'\s+([.,;:])', r'\1', merged)
    
    return merged.strip()

## 8. Main Extraction Pipeline

In [53]:
def extract_repository_metadata(repo_url: str, only_readme: bool = False) -> Dict[str, Any]:
    """Complete metadata extraction pipeline for a GitHub repository."""
    logger.info(f"Starting metadata extraction for: {repo_url}")
    
    # Step 1: Extract GitHub metadata
    logger.info("Step 1: Extracting GitHub metadata...")
    github_metadata = extract_github_metadata(repo_url)
    if not github_metadata:
        logger.error("Failed to extract GitHub metadata")
        return {}
    
    # Step 2: Extract from structured files
    logger.info("Step 2: Extracting structured files metadata...")
    if not only_readme:
        structured_files = github_metadata.pop('structured_files', {})
        structured_metadata = extract_from_structured_files(structured_files)
    
    # Step 3: Extract from README using NER model
    logger.info("Step 3: Extracting metadata from README using NER model...")
    readme_content : str = github_metadata.pop('readme_content', '')
    software_name: str = github_metadata.get('name', '')
    model_metadata = extract_metadata_from_readme(readme_content, software_name)

    # Preprocess model output: merge B-/I- labels and normalize URLs/requirements
    # Preprocess model output with enhanced processing
    try:
        base_codemeta = None # {**github_metadata, **structured_metadata}
        #logger.info(f"Base CodeMeta before model preprocessing: {base_codemeta}")
        model_metadata = preprocess_model_metadata(model_metadata, base_codemeta)
    except Exception as e:
        logger.warning(f"Model postprocessing failed: {e}")

 #   logger.info(f"Model extracted entities (postprocessed): {list(model_metadata.keys())}")
    ## show all extracted entity values for debugging
  #  for label, entities in model_metadata.items():
 #       logger.info(f"  {label}: {entities}")
    
    if only_readme:
        name: Dict[str, Any] = {"name": software_name} if software_name else {}
        model_metadata = {**create_codemeta_base(), **name, **model_metadata}
        return model_metadata
    
    # Step 4: Merge all metadata with priority
    logger.info("Step 4: Merging metadata with priority (GitHub > Structured > Model)...")
    final_metadata = merge_metadata_with_priority(github_metadata, structured_metadata, model_metadata)

    logger.info("Metadata extraction completed successfully")
    return final_metadata


def _convert_for_json(obj):
    """Recursively convert objects to JSON-serializable forms.

    - datetime.date and datetime.datetime -> ISO string
    - dict/list/tuple/set -> recursively convert
    - numpy scalars -> native python type (if numpy available)
    - everything else: returned as-is (json.dump will raise if still not serializable)
    """
    try:
        import datetime as _dt
    except Exception:
        _dt = None

    # None
    if obj is None:
        return None

    # Dates/datetimes
    if _dt and isinstance(obj, (_dt.date, _dt.datetime)):
        try:
            return obj.isoformat()
        except Exception:
            return str(obj)

    # Dicts
    if isinstance(obj, dict):
        return {str(k): _convert_for_json(v) for k, v in obj.items()}

    # Lists/tuples/sets
    if isinstance(obj, (list, tuple, set)):
        return [_convert_for_json(v) for v in obj]

    # Numpy scalars (optional)
    try:
        import numpy as _np
        if isinstance(obj, _np.generic):
            return obj.item()
    except Exception:
        pass

    # Fallback: primitive types (int/float/str/bool) will pass through
    return obj


def save_codemeta_json(metadata: Dict[str, Any], output_path: str) -> None:
    """Save metadata as CodeMeta JSON file, converting non-serializable objects.

    This will convert datetime objects to ISO strings to avoid
    "Object of type date is not JSON serializable" errors.
    """
    # Remove internal fields before saving
    clean_metadata = {k: v for k, v in metadata.items() if not k.startswith('_')}

    # Convert to JSON-safe structures
    serializable = _convert_for_json(clean_metadata)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(serializable, f, indent=2, ensure_ascii=False)

    logger.info(f"CodeMeta JSON saved to: {output_path}")


def display_metadata_summary(metadata: Dict[str, Any]) -> None:
    """Display a summary of extracted metadata."""
    print("\n" + "="*60)
    print("METADATA EXTRACTION SUMMARY")
    print("="*60)
    
    # Basic information
    print(f"Name: {metadata.get('name', 'N/A')}")

    # Make description safe to slice and measure
    desc = metadata.get('description', '')
    if desc is None:
        desc = ''
    if not isinstance(desc, str):
        try:
            desc = str(desc)
        except Exception:
            desc = ''
    short_desc = desc[:100]
    suffix = '...' if len(desc) > 100 else ''
    print(f"Description: {short_desc}{suffix}")

    print(f"Version: {metadata.get('version', 'N/A')}")
    print(f"License: {metadata.get('license', 'N/A')}")
    print(f"Programming Language: {metadata.get('programmingLanguage', 'N/A')}")
    
    # URLs
    if metadata.get('codeRepository'):
        print(f"Repository: {metadata['codeRepository']}")
    if metadata.get('url'):
        print(f"Homepage: {metadata['url']}")
    
    # Authors
    if metadata.get('author'):
        authors_val = metadata['author']
        # Ensure authors is a list for counting
        if not isinstance(authors_val, list):
            try:
                authors_list = list(authors_val)
            except Exception:
                authors_list = [authors_val]
        else:
            authors_list = authors_val

        print(f"\nAuthors ({len(authors_list)}):")  
        for author in authors_list[:3]:  # Show first 3
            print(f"  - {author.get('name', 'Unknown') if isinstance(author, dict) else str(author)}")
        if len(authors_list) > 3:
            print(f"  ... and {len(authors_list) - 3} more")
    
    # Keywords
    keywords = metadata.get('keywords')
    if keywords:
        if isinstance(keywords, list):
            kw_display = ', '.join([str(k) for k in keywords[:5]])
            more = len(keywords) - 5
        else:
            # If keywords is a single string or other type, coerce to string
            kw_str = str(keywords)
            kw_display = kw_str[:200]
            more = 0
        print(f"\nKeywords: {kw_display}")
        if more and more > 0:
            print(f"  ... and {more} more")
    
    # Model extracted entities
    if metadata.get('_modelExtracted'):
        print(f"\nModel Extracted Entities:")
        for label, entities in metadata['_modelExtracted'].items():
            try:
                count = len(entities)
            except Exception:
                # If entities isn't sized, coerce to list then count
                try:
                    count = len(list(entities))
                except Exception:
                    count = 1
            print(f"  {label}: {count} items")
    
    # Sources used
    if metadata.get('_sources'):
        sources = [k for k, v in metadata['_sources'].items() if v]
        print(f"\nData Sources: {', '.join(sources)}")
    
    print("="*60)


## 9. Execute Pipeline

In [ ]:
# Get GitHub repository URL from user
repo_url = input("Enter the GitHub repository URL: ").strip()

# Validate URL
if not re.match(r'https?://github\.com/[^/]+/[^/]+', repo_url):
    logger.error("Invalid GitHub repository URL format")
    print("Please provide a valid GitHub repository URL (e.g., https://github.com/owner/repo)")
else:
    try:
        # Extract repository metadata
        metadata = extract_repository_metadata(repo_url)
        
        if metadata:
            # Display summary
            display_metadata_summary(metadata)
            
            # Save as CodeMeta JSON
            repo_name = repo_url.split('/')[-1]
            output_path = f"output/{repo_name}_codemeta.json"
            save_codemeta_json(metadata, output_path)
            
            print(f"\n✅ Extraction completed successfully!")
            print(f"📄 CodeMeta JSON saved to: {output_path}")
            
        else:
            print("❌ Failed to extract metadata from the repository")
            
    except Exception as e:
        logger.error(f"Pipeline execution failed: {e}")
        print(f"❌ Error: {e}")

## 10. Batch Processing (Optional)

In [54]:
def batch_process_repositories(repo_urls: List[str], output_dir: str = "../evaluation/extrated_codemeta_files") -> None:
    """Process multiple repositories in batch."""
    os.makedirs(output_dir, exist_ok=True)
    
    results = {
        'successful': 0,
        'failed': 0,
        'errors': []
    }
    
    for i, repo_url in enumerate(repo_urls, 1):
        print(f"\n[{i}/{len(repo_urls)}] Processing: {repo_url}")
        
        try:
            metadata = extract_repository_metadata(repo_url, only_readme=True)
            
            if metadata:
                repo_name = repo_url.split('/')[-1]
                output_path = os.path.join(output_dir, f"{repo_name}_model_codemeta.json")
                save_codemeta_json(metadata, output_path)
                results['successful'] += 1
                print(f"✅ Success: {repo_name}")
            else:
                results['failed'] += 1
                results['errors'].append(f"{repo_url}: No metadata extracted")
                print(f"❌ Failed: {repo_url}")
                
        except Exception as e:
            results['failed'] += 1
            results['errors'].append(f"{repo_url}: {str(e)}")
            print(f"❌ Error: {repo_url} - {e}")
    
    # Summary
    print(f"\n{'='*60}")
    print("BATCH PROCESSING SUMMARY")
    print(f"{'='*60}")
    print(f"Total repositories: {len(repo_urls)}")
    print(f"Successful: {results['successful']}")
    print(f"Failed: {results['failed']}")
    print(f"Success rate: {results['successful']/len(repo_urls)*100:.1f}%")
    
    if results['errors']:
        print(f"\nErrors:")
        for error in results['errors'][:5]:  # Show first 5 errors
            print(f"  - {error}")
        if len(results['errors']) > 5:
            print(f"  ... and {len(results['errors']) - 5} more errors")

# Example batch processing (uncomment to use)


# 434_qc2 https://github.com/qc2nl/qc2
# 161_ReSurfEMG	 https://github.com/resurfemg-org/ReSurfEMG
# 276_Cesium  https://github.com/eWaterCycle/Cesium-NcWMS
# 121_admtools https://github.com/MindTheGap-ERC/admtools
# 451_APE. https://github.com/sanctuuary/APE
# 248_LUE. https://github.com/computationalgeography/lue
# 415_OpenSim_Creator. https://github.com/opensim-org/opensim-core
# 131_NNPDF. https://github.com/NNPDF/nnpdf
# 164_AutoPQ. https://github.com/SMEISEN/AutoPQ
# 324_iBridges-GUI. https://github.com/iBridges-for-iRODS/iBridges-GUI
# 418_byteparsing. https://github.com/parallelwindfarms/byteparsing

repo_urls = [
    "https://github.com/qc2nl/qc2",
    "https://github.com/resurfemg-org/ReSurfEMG",
    "https://github.com/eWaterCycle/Cesium-NcWMS",
    "https://github.com/MindTheGap-ERC/admtools",
    "https://github.com/sanctuuary/APE",
    "https://github.com/computationalgeography/lue",
    "https://github.com/opensim-org/opensim-core",
    "https://github.com/NNPDF/nnpdf",
    "https://github.com/SMEISEN/AutoPQ",
    "https://github.com/iBridges-for-iRODS/iBridges-GUI"
]

batch_process_repositories(repo_urls)

2025-11-18 13:45:06,081 - INFO - Starting metadata extraction for: https://github.com/qc2nl/qc2
2025-11-18 13:45:06,082 - INFO - Step 1: Extracting GitHub metadata...



[1/10] Processing: https://github.com/qc2nl/qc2


2025-11-18 13:45:09,084 - INFO - Step 2: Extracting structured files metadata...
2025-11-18 13:45:09,085 - INFO - Step 3: Extracting metadata from README using NER model...
2025-11-18 13:45:13,573 - INFO - Total entities extracted: 23
2025-11-18 13:45:13,580 - INFO - Raw model metadata before preprocess: {'SOFTWARE_REQUIREMENTS': ['pip', 'setuptools', 'pip', 'setuptools', 'prospector', 'isort', 'pip', 'setuptools', 'pip', 'setuptools', 'Qiskit Nature', 'PennyLane', 'cutter'], 'SOFTWARE_REQUIREMENTS_URL': ['.', 'https://pypi.org/project/prospector/', 'https://pycqa.github.io/isort/.', 'https://qiskit.org/ecosystem/nature/', 'https://pennylane.ai/.'], 'DOCUMENTATION_URL': ['://qc2.readthedocs.io/en/latest/?badge=latest', '://qc2.readthedocs.io/en/latest/?badge=latest', '://qc2.readthedocs.io', '://qc2.readthedocs.io'], 'CITATION_URL': ['https://doi.org/10.5281/zenodo.14186370'], '_entity_sequence': [('SOFTWARE_REQUIREMENTS', 'pip'), ('SOFTWARE_REQUIREMENTS', 'setuptools'), ('SOFTWARE_REQ

✅ Success: qc2

[2/10] Processing: https://github.com/resurfemg-org/ReSurfEMG


2025-11-18 13:45:16,010 - INFO - Step 2: Extracting structured files metadata...
2025-11-18 13:45:16,011 - INFO - Step 3: Extracting metadata from README using NER model...
2025-11-18 13:45:18,575 - INFO - Total entities extracted: 33
2025-11-18 13:45:18,582 - INFO - Raw model metadata before preprocess: {'CITATION_URL': ['https://doi.org/10.5281/zenodo.6811553', 'https://joss.theoj.org/papers/5f08d1f2bb717b7d05762296e37ded3d', 'https://zenodo.org/badge/latestdoi/635680008'], 'LICENSE_URL': ['https://opensource.org/licenses/Apache-2.0'], 'OPERATING_SYSTEM': ['Linux', 'Win', 'OSX', 'Linux', 'OSX', 'Windows', 'Linux', 'OSX', 'Windows', 'Linux', 'OSX', 'Windows', 'Linux', 'OSX', 'Windows', 'Linux', 'OSX', 'Windows'], 'SOFTWARE_REQUIREMENTS': ['Anaconda', 'conda', 'mamba', 'v', '3', 'v', 'upyter', 'y'], 'RUNTIME_PLATFORM': ['Python 3.9+'], 'DOCUMENTATION_URL': ['https://resurfemg-org.github.io/ReSurfEMG/'], 'LICENSE': ['Apache License, version 2.0'], '_entity_sequence': [('CITATION_URL', '

...

✅ Success: nnpdf

[9/10] Processing: https://github.com/SMEISEN/AutoPQ


2025-11-18 13:45:41,908 - INFO - Step 2: Extracting structured files metadata...
2025-11-18 13:45:41,909 - INFO - Step 3: Extracting metadata from README using NER model...
2025-11-18 13:45:43,212 - INFO - Total entities extracted: 11
2025-11-18 13:45:43,219 - INFO - Raw model metadata before preprocess: {'CITATION': ['.', 'Kaleb Phipps, Stefan Meisenbacher, Benedikt Heidrich, Marian Turowski, Ralf Mikut, and Veit Hagenmeyer. 2023. Loss-customised probabilistic energy time series forecasts using automated hyperparameter optimisation. In Proceedings of the 14th ACM International Conference on Future Energy Systems (e-Energy ’23), Association for Computing Machinery, New York, NY, USA, 271–286. [https://doi.org/10.1145/3575813.3595204] https://doi.org/10.1145/3575813.3595204', 'fan Meisenbacher et al. 2024. AutoPQ: Automated point forecast-based quantile forecasts. In preparation.', '.'], 'REFERENCE_PUBLICATION': ['B. Heidrich, M. Turowski, K. Phipps, K. Schmieder, W. Süß, R. Mikut, and 

✅ Success: AutoPQ

[10/10] Processing: https://github.com/iBridges-for-iRODS/iBridges-GUI


2025-11-18 13:45:45,203 - INFO - Step 2: Extracting structured files metadata...
2025-11-18 13:45:45,204 - INFO - Step 3: Extracting metadata from README using NER model...
2025-11-18 13:45:46,442 - INFO - Total entities extracted: 17
2025-11-18 13:45:46,449 - INFO - Raw model metadata before preprocess: {'CITATION_URL': ['https://doi.org/10.5281/zenodo.17139123'], 'OPERATING_SYSTEM': ['Windows', 'Mac OS', 'Linux', 'Windows', 'macOS', 'Ubuntu', 'Windows', 'Mac', 'Linux', 'Mac', 'Linux', 'Windows'], 'RUNTIME_PLATFORM': ['Python 3.9 or higher'], 'SOFTWARE_REQUIREMENTS_VERSION': ['.2.11 or higher', '.3.0 or higher'], 'LICENSE': ['LGPL license'], '_entity_sequence': [('CITATION_URL', 'https://doi.org/10.5281/zenodo.17139123'), ('OPERATING_SYSTEM', 'Windows'), ('OPERATING_SYSTEM', 'Mac OS'), ('OPERATING_SYSTEM', 'Linux'), ('RUNTIME_PLATFORM', 'Python 3.9 or higher'), ('SOFTWARE_REQUIREMENTS_VERSION', '.2.11 or higher'), ('SOFTWARE_REQUIREMENTS_VERSION', '.3.0 or higher'), ('OPERATING_SYSTEM

✅ Success: iBridges-GUI

BATCH PROCESSING SUMMARY
Total repositories: 10
Successful: 10
Failed: 0
Success rate: 100.0%
